In [ ]:
# ==========================================
# Archivo: ejercicio3_trashnet_cnn.py
# ==========================================
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing import image
import numpy as np
import urllib.request
import os

# Definición de las clases de TrashNet
clases_residuos = ['Cartón', 'Vidrio', 'Metal', 'Papel', 'Plástico', 'Basura']

# 2. Diseño de la Arquitectura CNN
modelo_cnn = Sequential([
    # Capa Convolucional 1 + Max Pooling
    Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)),
    MaxPooling2D((2, 2)),
    
    # Capa Convolucional 2 + Max Pooling
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    
    # Capa Convolucional 3 + Max Pooling
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    
    # Transición a Fully Connected
    Flatten(),
    
    # Reducción de sobreajuste y Capa Oculta Densa
    Dropout(0.5),
    Dense(128, activation='relu'),
    
    # 6. Capa de Salida (6 clases)
    Dense(6, activation='softmax')
])

# Compilación del modelo
modelo_cnn.compile(optimizer='adam', 
                   loss='sparse_categorical_crossentropy', 
                   metrics=['accuracy'])

print("=== Arquitectura de la CNN ===")
modelo_cnn.summary()

# Nota: El entrenamiento completo requiere descargar el dataset de Kaggle y montar un flujo de datos.
# history = modelo_cnn.fit(train_dataset, validation_data=val_dataset, epochs=20)

# 7. Prueba con una imagen nueva descargada de internet
def predecir_imagen_nueva(url, modelo):
    # Descargar la imagen
    nombre_archivo = "imagen_prueba.jpg"
    urllib.request.urlretrieve(url, nombre_archivo)
    
    # Preprocesamiento
    img = image.load_img(nombre_archivo, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0) # Añadir dimensión de batch
    img_array /= 255.0 # Normalización
    
    # Predicción
    prediccion = modelo.predict(img_array)
    indice_clase = np.argmax(prediccion[0])
    confianza = np.max(prediccion[0]) * 100
    
    print(f"\nResultado de la predicción:")
    print(f"La imagen corresponde a: {clases_residuos[indice_clase]} (Confianza: {confianza:.2f}%)")
    
    os.remove(nombre_archivo) # Limpieza

# URL de ejemplo de una botella de plástico (simulación)
url_ejemplo = "https://upload.wikimedia.org/wikipedia/commons/4/41/Plastic_bottle.jpg"
print("\nBajando nueva imagen y realizando inferencia...")
try:
    predecir_imagen_nueva(url_ejemplo, modelo_cnn)
except Exception as e:
    print("Ocurrió un error al descargar o procesar la imagen:", e)